In [ ]:
import pandas as pd

df = pd.read_csv('mm_master_demos.csv')
print(df.shape)
print(df.dtypes)
df.head()

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nNumerical columns:", df.select_dtypes(include='number').shape[1])
print("Categorical columns:", df.select_dtypes(include='object').shape[1])
print("\n", df.describe())

# CS:GO Competitive Matchmaking EDA

## Dataset Overview
This dataset contains 955,466 damage entries from competitive CS:GO matchmaking games.
It has 33 columns total — 18 numerical and 14 categorical.

**Source:** Kaggle — CS:GO Competitive Matchmaking Data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numerical_cols = ['hp_dmg', 'arm_dmg', 'round', 'seconds', 
                  'att_rank', 'vic_rank', 'ct_eq_val', 't_eq_val', 'avg_match_rank']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='black')
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.suptitle('Distributions of Numerical Variables', fontsize=16)
plt.tight_layout()
plt.show()

## Section 1 — 1D Distributions: Numerical Variables

**hp_dmg (Health Damage):**
Min=0, Max=100, Mean=27.3. The distribution is multimodal with peaks around 
10-15 and 25-30 damage, reflecting common weapon damage values. The spike at 
100 represents instant kills. Low damage hits (0-5) are likely from shots that 
barely connected.

**arm_dmg (Armor Damage):**
Min=0, Max=100, Mean=3.7. Heavily right-skewed — the vast majority of damage 
entries show 0 armor damage, meaning most victims either had no armor or were 
hit in unarmored areas like the head or legs.

**round:**
Min=1, Max=30, Mean=13.5. Fairly uniform distribution across all 30 rounds 
with a slight drop-off in later rounds, suggesting some matches end before 
reaching round 30.

**seconds:**
Min=96, Max=3544, Mean=1342. Bimodal shape with peaks around 500s and 1500s, 
reflecting the two halves of a CS:GO match. Outliers beyond 3000s represent 
unusually long matches.

**att_rank / vic_rank:**
Both centered around ranks 10-13 (Gold Nova to Master Guardian range), which 
is the most populated skill bracket in CS:GO matchmaking. Very few entries 
below rank 5 or above rank 16.

**ct_eq_val / t_eq_val (Equipment Value):**
Both show bimodal distributions — one peak near $0-2000 (eco/save rounds) and 
another around $20,000-27,000 (full buy rounds). This reflects CS:GO's 
economy system where teams either save money or go for full buys.

**avg_match_rank:**
Discrete distribution clustered between ranks 8-14, confirming most matches 
are played at mid-tier skill levels.

In [ ]:
categorical_cols = ['map', 'att_side', 'hitbox', 'wp_type', 'winner_side']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    counts = df[col].value_counts()
    axes[i].bar(counts.index, counts.values, color='steelblue', edgecolor='black')
    axes[i].set_title(col)
    axes[i].set_xlabel('Category')
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)

axes[5].set_visible(False)
plt.suptitle('Distributions of Categorical Variables', fontsize=16)
plt.tight_layout()
plt.show()

## Section 1 — Categorical Variable Distributions

**map:**
de_mirage is by far the most played map (~260,000 damage entries), followed by
de_dust2 and de_cache. This matches real-world CS:GO popularity where Mirage
has historically been the most picked competitive map. Many community maps
appear rarely, representing outliers in the dataset.

**att_side (Attacker Side):**
Nearly equal split between CounterTerrorist (~490,000) and Terrorist (~460,000)
damage entries. This makes sense as both sides deal damage every round —
the slight CT edge may reflect CT-sided maps like Mirage dominating the dataset.

**hitbox:**
Chest is the most common hit location by a large margin (~370,000), followed
by Stomach and Head. This reflects realistic aim patterns — body shots are
more common than headshots in real gameplay. Arm and leg hits are the rarest,
suggesting players rarely aim for limbs intentionally.

**wp_type (Weapon Type):**
Rifles dominate (~415,000) as the primary damage dealer, followed by Pistols
and SMGs. Grenades, Snipers, and Heavy weapons are used far less frequently.
This confirms rifles as the meta weapon class in competitive CS:GO.

**winner_side:**
Nearly even split between Terrorist and CounterTerrorist wins, confirming
the dataset is well-balanced across both sides with no significant bias.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

df[numerical_cols].plot(kind='box', ax=ax, patch_artist=True)
ax.set_title('Side-by-Side Boxplot of All Numerical Variables')
ax.set_xlabel('Variable')
ax.set_ylabel('Value')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Section 1 — Combined Boxplot

The side-by-side boxplot reveals the vastly different scales across variables.
ct_eq_val and t_eq_val have the largest spread ($0–$38,000) reflecting CS:GO's
economy system. Most other variables like hp_dmg, round, and ranks are
compressed near zero by comparison. Notable outliers appear in hp_dmg and
arm_dmg at their maximum value of 100, and in att_rank/vic_rank where a small
number of very low-ranked players appear in otherwise mid-ranked matches.

In [ ]:
sample = df[numerical_cols].sample(2000, random_state=42)

sns.pairplot(sample, plot_kws={'alpha': 0.3, 'size': 1})
plt.suptitle('Pairwise Scatter Plot Matrix', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

## Section 2 — Pairwise Scatter Plot Matrix

The most striking relationship is between **round and seconds** — they show 
a strong positive linear correlation (visible diagonal line), which makes 
perfect sense since later rounds occur later in match time.

**hp_dmg vs arm_dmg** shows a weak positive relationship — higher damage hits 
tend to also deal more armor damage, but many hits deal HP damage with zero 
armor damage (unarmored players or headshots).

**att_rank vs vic_rank** shows a strong positive correlation — players tend 
to fight opponents of similar rank, confirming matchmaking is working as 
intended.

**ct_eq_val vs t_eq_val** shows a moderate positive correlation — both teams 
tend to buy or save together in the same round, reflecting the mirrored 
economy system in CS:GO.

Most other variable pairs show no clear relationship (scattered clouds), 
meaning damage dealt is largely independent of which round or map time it 
occurred in.

In [ ]:
# Map vs Winner Side aggregated
ct_table = pd.crosstab(df['map'], df['winner_side'], normalize='index') * 100
ct_table = ct_table[ct_table.sum(axis=1) > 0].head(10)

ct_table.plot(kind='bar', figsize=(14, 6), colormap='Set2', edgecolor='black')
plt.title('Win Rate by Map (CT vs T) — Top 10 Maps')
plt.xlabel('Map')
plt.ylabel('Win Rate (%)')
plt.xticks(rotation=45)
plt.legend(title='Winner Side')
plt.tight_layout()
plt.show()

## Section 3 — Categorical vs Categorical

To show the relationship between two categorical variables (map and winner_side),
we aggregated win rates using a crosstab and displayed them as a grouped bar chart.

**Key findings:**
- **cs_insertion** is heavily CT-sided (~70% CT win rate) — one of the most 
  imbalanced maps in the dataset
- **de_austria** is heavily T-sided (~70% T win rate) — the opposite extreme
- **de_aztec** also favors CT significantly (~67% CT wins)
- **cs_agency, cs_assault, de_canals** are nearly 50/50 — very balanced maps
- **cs_italy and cs_office** favor Terrorists noticeably

This is genuinely useful CS:GO insight — map design directly determines which 
side has the structural advantage, and this data confirms what the community 
has known anecdotally about map balance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# hp_dmg by attacker side
sns.boxplot(data=df, x='att_side', y='hp_dmg', ax=axes[0], palette='Set2')
axes[0].set_title('HP Damage by Attacker Side')
axes[0].set_xlabel('Attacker Side')
axes[0].set_ylabel('HP Damage')

# hp_dmg by weapon type
sns.boxplot(data=df, x='wp_type', y='hp_dmg', ax=axes[1], palette='Set2')
axes[1].set_title('HP Damage by Weapon Type')
axes[1].set_xlabel('Weapon Type')
axes[1].set_ylabel('HP Damage')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Categorical vs Numerical Relationships', fontsize=16)
plt.tight_layout()
plt.show()

## Section 4 — Categorical vs Numerical

Boxplots are used to show how a numerical variable (hp_dmg) is distributed 
across different categories. Color differentiates the categories visually.

**HP Damage by Attacker Side:**
CTs and Ts deal very similar damage overall — median around 22-25 HP per hit.
Terrorists show a slightly wider spread and higher median, suggesting T-side 
weapons may deal marginally more damage per hit on average. Both sides have 
outliers reaching 100 (instant kills).

**HP Damage by Weapon Type:**
Snipers deal by far the highest median damage (~90 HP) with a very tight 
box — they consistently deal massive damage, often one-shotting enemies.
Grenades and SMGs deal the lowest median damage per hit (~5-18 HP).
Rifles sit in the middle (~25 HP median) but are the most common weapon.
Equipment and Heavy weapons show wide variance — inconsistent damage output.
The Unknown category likely represents edge cases or data entry issues.